# Bulk RNA-seq limma-voom Differential Expression Template

本 notebook 使用 `edgeR` + `limma-voom` 进行 bulk RNA-seq 差异表达分析，适用于复杂设计矩阵、批次校正或多组比较场景。可作为 `RNAseq_General.ipynb`（DESeq2 流程）的替代或补充。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
INPUT_FILE <- "./0-Data/counts.tsv"
INPUT_FORMAT <- "tsv"                        # "tsv", "csv", or "excel"
GENE_NAME_COL <- "gene_name"
BIOTYPE_COL <- "gene_biotype"
BIOTYPE_FILTER <- "protein_coding"
COUNT_COLS <- NULL                          # NULL = auto-detect numeric columns

SAMPLE_NAMES <- c(
  "Control_1", "Control_2", "Control_3",
  "Treatment_1", "Treatment_2", "Treatment_3"
)
GROUPS <- c(rep("Control", 3), rep("Treatment", 3))
GROUP_LEVELS <- c("Control", "Treatment")

# Comparisons: list of c(name, numerator, denominator)
COMPARISONS <- list(
  c("Treatment_vs_Control", "Treatment", "Control")
)

# Optional batch correction (must be same length as SAMPLE_NAMES)
BATCH_VECTOR <- NULL                         # e.g. c("B1","B1","B2","B2","B1","B2"); NULL = no correction

# Thresholds
DEG_PADJ_CUTOFF <- 0.05
DEG_LFC_CUTOFF <- 0.5
MIN_COUNT <- 10
MIN_SAMPLE_FRAC <- 0.5

# Output
OUTDIR <- "RNAseq_limma_voom_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(OUTDIR, "1-DEG"), showWarnings = FALSE)
dir.create(file.path(OUTDIR, "2-GSEA"), showWarnings = FALSE)
dir.create(file.path(OUTDIR, "3-Visualization"), showWarnings = FALSE)


## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse", "RColorBrewer"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("edgeR", "limma", "sva", "clusterProfiler", "org.Mm.eg.db"))

suppressPackageStartupMessages({
  library(tidyverse)
  library(edgeR)
  library(limma)
  library(sva)
  library(clusterProfiler)
  library(org.Mm.eg.db)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "io_utils.R"))
source(file.path(LIB_DIR, "deg_utils.R"))
source(file.path(LIB_DIR, "enrichment_utils.R"))
source(file.path(LIB_DIR, "limma_voom_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")


## 3. Load Counts and Build Design Matrix

In [ ]:
rawcount <- read_count_table(INPUT_FILE, INPUT_FORMAT)
if (!is.null(BIOTYPE_COL) && BIOTYPE_COL %in% colnames(rawcount)) {
  rawcount <- rawcount[rawcount[[BIOTYPE_COL]] == BIOTYPE_FILTER, ]
}
count_col_names <- detect_count_columns(rawcount, GENE_NAME_COL, COUNT_COLS)
validate_sample_design(SAMPLE_NAMES, GROUPS, GROUP_LEVELS, COMPARISONS, count_col_names)

countData <- build_count_matrix(rawcount, GENE_NAME_COL, count_col_names, SAMPLE_NAMES)
group <- factor(GROUPS, levels = GROUP_LEVELS)
design <- make_group_design(group)
cat("Design matrix:\n")
print(head(design))


## 4. limma-voom Pipeline

In [ ]:
dge <- prepare_dge_for_voom(countData, group = group,
                             min_counts_per_sample = MIN_COUNT,
                             min_sample_frac = MIN_SAMPLE_FRAC)
v <- run_voom(dge, design = design, plot_file = file.path(OUTDIR, "3-Visualization", "voom_mean_variance_trend.pdf"))

# Optional batch correction
batch_vec <- if (!is.null(BATCH_VECTOR) && length(BATCH_VECTOR) == length(SAMPLE_NAMES)) BATCH_VECTOR else NULL
if (!is.null(BATCH_VECTOR) && is.null(batch_vec)) {
  warning("BATCH_VECTOR length does not match SAMPLE_NAMES; skipping batch correction.")
}
if (!is.null(batch_vec)) {
  v <- remove_batch_effect_voom(v, batch = batch_vec, design = design)
  cat("Batch correction applied with batch vector:", paste(unique(batch_vec), collapse = ", "), "\n")
}

res_list <- run_limma_contrasts(v, design, COMPARISONS)
deg_summary <- write_limma_results(res_list, outdir = file.path(OUTDIR, "1-DEG"))
print(deg_summary)


## 5. Visualization

In [ ]:
group_colors <- make_group_colors(GROUP_LEVELS)

# Volcano plots
for (comp_name in names(res_list)) {
  plot_volcano_pdf(
    res_list[[comp_name]], comp_name = comp_name,
    pvalue_thresh = DEG_PADJ_CUTOFF, log2fc_thresh = DEG_LFC_CUTOFF,
    filename = file.path(OUTDIR, "3-Visualization", paste0("Volcano_", comp_name, ".pdf")),
    pvalue_column = "padj", lfc_column = "log2FoldChange"
  )
}

# DEG summary barplot
p_deg <- plot_deg_summary_pdf(
  deg_summary,
  filename = file.path(OUTDIR, "3-Visualization", "DEG_summary.pdf")
)
print(p_deg)

# Top DEG heatmap
all_sig_genes <- unique(unlist(lapply(res_list, function(res) {
  res |>
    filter(!is.na(padj), padj < DEG_PADJ_CUTOFF, abs(log2FoldChange) > DEG_LFC_CUTOFF) |>
    pull(gene_name)
})))
if (length(all_sig_genes) >= 2) {
  mat <- v$E[intersect(all_sig_genes, rownames(v$E)), ]
  plot_expression_heatmap_pdf(
    mat,
    filename = file.path(OUTDIR, "3-Visualization", "DEG_heatmap.pdf"),
    title = "limma-voom DEGs",
    group = GROUPS, group_levels = GROUP_LEVELS, group_colors = group_colors,
    width = 8, height = 10
  )
}


## 6. ORA and GSEA

In [ ]:
species_org <- org.Mm.eg.db
organism_code <- "mmu"

universe <- map_symbols_to_entrez(rownames(countData), species_org)$ENTREZID

go_ora_map <- list()
kegg_ora_map <- list()
go_gsea_map <- list()
kegg_gsea_map <- list()

for (comp_name in names(res_list)) {
  genes <- genes_for_enrichment(
    res_list[[comp_name]],
    pvalue_thresh = DEG_PADJ_CUTOFF, log2fc_thresh = DEG_LFC_CUTOFF,
    pvalue_column = "padj", lfc_column = "log2FoldChange"
  )
  ego <- run_go_ora(genes$sig, org_db = species_org, universe = universe)
  ekegg <- run_kegg_ora(genes$sig, org_db = species_org, universe = universe, organism = organism_code)
  if (!is.null(ego)) {
    write.csv(as.data.frame(ego), file.path(OUTDIR, "2-GSEA", paste0("GO_ORA_", comp_name, ".csv")), row.names = FALSE)
    plot_enrich_suite_pdf(ego, file.path(OUTDIR, "3-Visualization", paste0("GO_ORA_", comp_name)), paste("GO ORA", comp_name))
    go_ora_map[[comp_name]] <- ego
  }
  if (!is.null(ekegg)) {
    write.csv(as.data.frame(ekegg), file.path(OUTDIR, "2-GSEA", paste0("KEGG_ORA_", comp_name, ".csv")), row.names = FALSE)
    plot_enrich_suite_pdf(ekegg, file.path(OUTDIR, "3-Visualization", paste0("KEGG_ORA_", comp_name)), paste("KEGG ORA", comp_name))
    kegg_ora_map[[comp_name]] <- ekegg
  }

  ranked <- ranked_gene_list_limma(res_list[[comp_name]], rank_column = "t")
  entrez_ranked <- make_entrez_ranked_list(ranked, species_org)
  gsea_go <- run_go_gsea(entrez_ranked, org_db = species_org)
  gsea_kegg <- run_kegg_gsea(entrez_ranked, organism = organism_code)
  if (!is.null(gsea_go)) {
    write.csv(as.data.frame(gsea_go), file.path(OUTDIR, "2-GSEA", paste0("GO_GSEA_", comp_name, ".csv")), row.names = FALSE)
    plot_gsea_suite_pdf(gsea_go, file.path(OUTDIR, "3-Visualization", paste0("GO_GSEA_", comp_name)), paste("GO GSEA", comp_name))
    go_gsea_map[[comp_name]] <- gsea_go
  }
  if (!is.null(gsea_kegg)) {
    write.csv(as.data.frame(gsea_kegg), file.path(OUTDIR, "2-GSEA", paste0("KEGG_GSEA_", comp_name, ".csv")), row.names = FALSE)
    plot_gsea_suite_pdf(gsea_kegg, file.path(OUTDIR, "3-Visualization", paste0("KEGG_GSEA_", comp_name)), paste("KEGG GSEA", comp_name))
    kegg_gsea_map[[comp_name]] <- gsea_kegg
  }
}


### 6.1 Publication-Grade Theme Dot-heatmap

Group ORA/GSEA terms into biological themes for a manuscript-ready overview.

In [ ]:
theme_outdir <- file.path(OUTDIR, "3-Visualization", "ThemeEnrichment")
dir.create(theme_outdir, showWarnings = FALSE, recursive = TRUE)
theme_defs <- default_enrichment_themes()

if (length(go_ora_map) > 0) {
  p <- plot_theme_dotheatmap_from_results(go_ora_map, file.path(theme_outdir, "Theme_dotheatmap_GO_ORA.pdf"),
    title = "GO ORA Biological Themes", subtitle = paste("GO-BP ORA |", DEG_PADJ_CUTOFF, "& |log2FC| >", DEG_LFC_CUTOFF), theme_defs = theme_defs, ontology_filter = "BP")
  if (!is.null(p)) print(p)
}
if (length(kegg_ora_map) > 0) {
  p <- plot_theme_dotheatmap_from_results(kegg_ora_map, file.path(theme_outdir, "Theme_dotheatmap_KEGG_ORA.pdf"),
    title = "KEGG ORA Pathway Themes", subtitle = paste("KEGG ORA |", DEG_PADJ_CUTOFF, "& |log2FC| >", DEG_LFC_CUTOFF), theme_defs = theme_defs, ontology_filter = NULL)
  if (!is.null(p)) print(p)
}
if (length(go_gsea_map) > 0) {
  p <- plot_theme_dotheatmap_from_results(go_gsea_map, file.path(theme_outdir, "Theme_dotheatmap_GO_GSEA.pdf"),
    title = "GO GSEA Biological Themes", subtitle = "GO-BP GSEA", theme_defs = theme_defs, ontology_filter = "BP")
  if (!is.null(p)) print(p)
}
if (length(kegg_gsea_map) > 0) {
  p <- plot_theme_dotheatmap_from_results(kegg_gsea_map, file.path(theme_outdir, "Theme_dotheatmap_KEGG_GSEA.pdf"),
    title = "KEGG GSEA Pathway Themes", subtitle = "KEGG GSEA", theme_defs = theme_defs, ontology_filter = NULL)
  if (!is.null(p)) print(p)
}


### 6.2 Publication-Grade Single-Term GSEA Figures

Generate one running-enrichment figure per selected GSEA term.

In [ ]:
single_term_outdir <- file.path(OUTDIR, "3-Visualization", "ThemeEnrichment", "single_term_gsea")
dir.create(single_term_outdir, showWarnings = FALSE, recursive = TRUE)

for (comp_name in names(res_list)) {
  ggo <- go_gsea_map[[comp_name]]
  gkegg <- kegg_gsea_map[[comp_name]]

  if (!is.null(ggo) && nrow(as.data.frame(ggo)) > 0) {
    ggo_df <- as.data.frame(ggo)
    ggo_df <- ggo_df[order(ggo_df$p.adjust, -abs(ggo_df$NES)), ]
    top_terms <- rbind(utils::head(ggo_df[ggo_df$NES > 0, ], 3),
                       utils::head(ggo_df[ggo_df$NES < 0, ], 3))
    plot_gsea_term_figures_from_df(ggo, top_terms,
      outdir = file.path(single_term_outdir, paste0("GO_", comp_name)),
      contrast_label = comp_name, prefix = "gseaplot2_GO")
  }

  if (!is.null(gkegg) && nrow(as.data.frame(gkegg)) > 0) {
    gkegg_df <- as.data.frame(gkegg)
    gkegg_df <- gkegg_df[order(gkegg_df$p.adjust, -abs(gkegg_df$NES)), ]
    top_terms <- rbind(utils::head(gkegg_df[gkegg_df$NES > 0, ], 3),
                       utils::head(gkegg_df[gkegg_df$NES < 0, ], 3))
    plot_gsea_term_figures_from_df(gkegg, top_terms,
      outdir = file.path(single_term_outdir, paste0("KEGG_", comp_name)),
      contrast_label = comp_name, prefix = "gseaplot2_KEGG")
  }
}


## 7. Save Session

In [ ]:
save.image(file = file.path(OUTDIR, "limma_voom_workspace.Rdata"))
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
cat("Analysis complete. Outputs saved to", OUTDIR, "\n")
